# Phase 2 — Generative AI: Model Setup and Prompt Templates

**SWE485 | Group 4 — University Student Retention & Advisory Dashboard**  
**Scope of this notebook so far:** Gemini setup, prompt-template documentation, and the shared generation interface.  
**Evaluation owner handoff:** The testing, qualitative/quantitative comparison, final template selection, ethics analysis, and lessons learned will be added by the evaluation teammate.

> **Quota and privacy:** All code cells in this initial notebook are offline: they do **not** send requests to Gemini. Do not paste API keys, personally identifying student data, or real student records into the notebook or GitHub.

## 1. Generative AI Model Setup

### 1.1 Objective and model

Google Gemini is used to turn structured student information into supportive, human-reviewable academic guidance. The implementation is in `Generative_AI/genai_utils.py` and exposes a single interface:

```python
generate_briefing(student, prediction, probs, top_features, cluster_profile, template)
```

The configured model is **`gemini-3.8-flash`**, matching the team's current implementation. The model name is read from the source module rather than being overridden in this notebook, to keep the evaluation consistent.

**Inputs:** early-stage student records; supervised-model predicted status and class probabilities; globally important model features with available student values; optional cluster profile; and template ID (`advisor` or `student`).  
**Output:** a generated English briefing or outreach message for review by a human advisor.

### 1.2 Dependencies and secure configuration

The repository's `requirements.txt` includes `google-genai` and `python-dotenv`. Install the project requirements in your Python environment if necessary:

```bash
python -m pip install -r requirements.txt
```

For local use, put `GEMINI_API_KEY=...` in a `.env` file in the **project root**. The implementation loads this file with `python-dotenv`. Never commit `.env` or reveal its contents. Deployment secrets should likewise be configured outside the repository.

The following cell locates the project root, imports the **existing** implementation, and reports only whether a key is configured. It does not display the key or make an API request.

In [ ]:
import os
import sys
from pathlib import Path

# Works when Jupyter starts in either the project root or Generative_AI/.
search_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (p for p in search_roots if (p / "Generative_AI" / "genai_utils.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the project root or Generative_AI/ folder."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Generative_AI.genai_utils import (
    MODEL_NAME,
    PROMPT_TEMPLATES,
    generate_briefing,
)

print("Configured Gemini model:", MODEL_NAME)
print("Gemini API key configured:", bool(os.getenv("GEMINI_API_KEY")))
print("Available templates:", ", ".join(PROMPT_TEMPLATES))
print("No Gemini API request was made.")

### 1.3 Implementation flow

1. The dashboard passes early-stage student data, a prediction and probabilities, available model indicators, and an optional cluster profile to `generate_briefing(...)`.
2. The function validates the template ID and checks that `GEMINI_API_KEY` is available.
3. It serializes the input as structured JSON and combines it with the selected prompt template and evidence/safety instructions.
4. The Gemini API returns generated text, which the dashboard presents for advisor review.

**Safeguards in the implementation:** The prompts distinguish predictions from facts, reject invented attendance or financial details, prohibit causal claims from global feature importance, and prohibit invented cluster characteristics when `cluster_profile` is `None`. Second-semester and target-label information must be excluded upstream before sending student records to Gemini. The generated text still requires human review; prompt instructions do not guarantee perfect compliance.

## 2. Prompt Engineering and Template Design

Two audience-specific templates are defined in `PROMPT_TEMPLATES` in `Generative_AI/genai_utils.py`. Both use the same structured inputs and evidence constraints; they differ in tone, detail, and intended action. Detailed documentation is maintained in `Generative_AI/prompt_design_rationale.md`.

| Attribute | Template 1: Advisor briefing | Template 2: Student outreach |
|---|---|---|
| Template ID | `advisor` | `student` |
| Audience | Academic advisors | Students |
| Tone | Professional, analytical | Warm, encouraging |
| Focus | Predicted status, observed indicators, supportive interventions | Practical next steps without alarming labels |
| Intended length | Approximately 120–160 words | Approximately 90–130 words |
| Human review | Before advisor action | Before student-facing delivery |

**Shared input fields:** `student`, `prediction`, `probs`, `top_features`, `cluster_profile`. The model's JSON representation uses the keys `student`, `predicted_status`, `class_probabilities`, `important_features`, and `cluster_profile`.

### 2.1 Template 1 — Advisor Briefing (`advisor`)

**Use case:** Help an academic advisor review predicted risk and consider supportive interventions.  
**Design rationale:** Advisors need concise, evidence-based summaries that distinguish model predictions from observed records.  
**Expected structure:** Risk/status summary, relevant recorded indicators and available cluster context, and 2–3 practical support actions.  
**Assumptions and limits:** Predictions may be wrong; global feature importance does not explain an individual prediction or establish causality; cluster profiles are general patterns, not individual diagnoses.

The next cell prints the exact template currently used by the application, without calling Gemini.

In [ ]:
print(PROMPT_TEMPLATES["advisor"].strip())

### 2.2 Template 2 — Student Outreach (`student`)

**Use case:** Draft a supportive student-facing message encouraging academic assistance.  
**Design rationale:** Student communication should be accessible, respectful, and encouraging, avoiding technical machine-learning terminology and deterministic dropout labels.  
**Expected structure:** Warm check-in, 2–3 realistic next steps, and an invitation to contact an advisor or student services.  
**Assumptions and limits:** Do not reveal sensitive predictions unnecessarily; do not invent personal circumstances, grades, attendance, or financial balances; an advisor must review the message before sending.

The next cell prints the exact template currently used by the application, without calling Gemini.

In [ ]:
print(PROMPT_TEMPLATES["student"].strip())

### 2.3 Structured input and function usage (offline illustration)

The following **synthetic** example shows the input contract for both templates. It is not a real student and does not call the API. For actual use, the dashboard must pass only approved early-stage features and the model's computed predictions/probabilities; cluster information may be `None` until clustering integration is complete.

In [ ]:
import json

# Synthetic example only — not a record from the student dataset.
example_student = {
    "Curricular units 1st sem (enrolled)": 6,
    "Curricular units 1st sem (approved)": 2,
    "Curricular units 1st sem (evaluations)": 4,
    "Tuition fees up to date": 1,
    "academic_progress": 2 / 6,
}
example_prediction = "Dropout"
example_probs = {"Dropout": 0.76, "Graduate": 0.15, "Enrolled": 0.09}
example_top_features = {"Curricular units 1st sem (approved)": 2}
example_cluster_profile = None

example_payload = {
    "student": example_student,
    "predicted_status": example_prediction,
    "class_probabilities": example_probs,
    "important_features": example_top_features,
    "cluster_profile": example_cluster_profile,
}
print(json.dumps(example_payload, indent=2))
print("No Gemini API request was made.")

### 2.4 Calling the implementation (reference only — **do not run during setup**)

The example below documents how the dashboard or the evaluation notebook can invoke either template. It is intentionally shown as **documentation, not an executable code cell**, to avoid using the shared API quota.

```python
advisor_text = generate_briefing(
    student=example_student,
    prediction=example_prediction,
    probs=example_probs,
    top_features=example_top_features,
    cluster_profile=example_cluster_profile,
    template="advisor",
)

student_text = generate_briefing(
    student=example_student,
    prediction=example_prediction,
    probs=example_probs,
    top_features=example_top_features,
    cluster_profile=example_cluster_profile,
    template="student",
)
```

**Important:** Each actual call consumes API quota and may encounter rate limits or temporary service errors (e.g., HTTP 503). The evaluation owner should control when these calls are run and how retries are handled. Do not change `MODEL_NAME` while the team's comparative evaluation is underway.

### 2.5 Offline interface checks

These checks validate the two template IDs and callable function without sending student data or API requests. They do **not** substitute for the required qualitative and quantitative evaluation.

In [ ]:
assert MODEL_NAME == "gemini-3.8-flash", "Coordinate with the team before changing the model."
assert {"advisor", "student"}.issubset(PROMPT_TEMPLATES)
assert all(PROMPT_TEMPLATES[name].strip() for name in ("advisor", "student"))
assert callable(generate_briefing)
print("Offline interface checks passed; no API requests were made.")

## 3. Handoff for Team Evaluation (to be completed by Sarah)

This notebook currently documents the **model setup and two prompt templates only**. The remaining project requirements are intentionally **not reported as completed** here:

- Run at least three test cases per template (the evaluation teammate is preparing five real test scenarios), recording inputs, outputs, and any retries or rate-limit errors.
- Compare outputs qualitatively (relevance, completeness, clarity, personalization, factual accuracy/safety) and quantitatively (e.g., length, keywords, readability, preference).
- Select and justify the most suitable template for the target use case, or explain why both remain useful for different audiences.
- Add integration strategy, ethical considerations, lessons learned, and references as required for the final Phase 2 deliverable.

**Related files:** `Generative_AI/genai_utils.py`, `Generative_AI/prompt_design_rationale.md`, and the dashboard's `pages/3_AI_Briefings.py`.  
**Privacy reminder:** Remove direct identifiers and secrets from all notebook outputs before committing. Review AI-generated messages before taking action or contacting students.